##### OpenTelemetry

OpenTelemetry, commonly called OTel, is an open-source observability framework used to generate and collect telemetry from applications.

Telemetry mainly means:
1. Traces
2. Metrics
3. Logs

#### Example why we need the Open Telemetry

- suppose we have built the python AI Application

``` markdown
Python AI Application
       ↓
   User Question
       ↓
   LLM Agent
       ↓
   OpenAI API
       ↓
   SQL Tool
       ↓
   Database
       ↓
   Final Answer
```

without instrumentation we might only see 
**Application Failed** 

with open telemetry, we can potentially see:

``` markdown
Trace ID: abc123

Agent              3.2 sec
│
├── LLM Call       1.8 sec
│   ├── Model      gpt-...
│   ├── Tokens     1,250
│
├── SQL Tool       0.9 sec
│
└── Final LLM      0.5 sec
```

This gives us much more information about what happened.


##### OpenTelemetry Architecture

``` markdown
             Your Application
                    │
                    ↓
             Instrumentation
                    │
                    ↓
             OpenTelemetry SDK
                    │
                    ↓
          OpenTelemetry Collector
                    │
                    ↓
                Exporter
                    │
       ┌────────────┼────────────┐
       ↓            ↓            ↓
    Backend 1    Backend 2    Backend 3
```

#### Instrumentation

Instrumentation means adding observability capabilities to your application so that it produces telemetry.

**For a system to be observable, it must be instrumented: that is, code from the system’s components must emit signals, such as traces, metrics, and logs.**

Using OpenTelemetry, you can instrument your code in two primary ways:

1. Code-based Solutions
2. Zero-Code Solutions


- Code-based solutions allow you to get deeper insight and rich telemetry from your application itself. They let you use the OpenTelemetry API to generate telemetry from your application, which acts as an essential complement to the telemetry generated by zero-code solutions.

- Zero-code solutions are great for getting started, or when you can’t modify the application you need to get telemetry out of. They provide rich telemetry from libraries you use and/or the environment your application runs in. Another way to think of it is that they provide information about what’s happening at the edges of your application

##### Example
Suppose we have 

``` python
def ask_llm(question):
    response = client.chat(....)
    return response
```

without instrumentation 

``` markdown
question
   ↓
LLM
   ↓
response
```

we dont automatically know:
- How long it took
- which model was used
- how many tokens were consumed 
- wether it failed
- which request triggered it

instrumentation adds observability around that operation.

Conceptually:

``` python
start_span()

response = client.chat(...)

record_attributes(...)
record_tokens(...)
record_latency(...)

end_span()
```



#### Spans

spans represents an operation.

example:

``` markdown
Trace
│
└── AI Request
      │
      ├── Retrieval
      │
      ├── LLM Call
      │
      └── Tool Call
```

Each operation can become an span.


* Span Aanatomy
``` markdown
Span
├── Trace ID
├── Span ID
├── Parent Span ID
├── Name
├── Start time
├── End time
├── Status
├── Attributes
├── Events
└── Links
```

Example:

``` markdown
Span Name:
llm.generate

Trace ID:
abc123

Span ID:
span789

Parent:
agent456

Duration:
1.8 seconds

Status:
OK
```


##### Span Attributes

Attributes provide additional information about a span.

Think:

``` markdown
Span = operation

Attributes = information describing that operation
```

``` markdown
example:

Span:
LLM Call

Attributes:
model = "..."
provider = "..."
temperature = 0.2
input_tokens = 500
output_tokens = 200
```

conceptually:

``` python
span.set_attribute("model", "...")
span.set_attribute("temperature", 0.2)
```

#### Span Events

``` markdown

Span: LLM Call
│
├── Event: request_started
│
├── Event: retry
│
├── Event: response_received
│
└── Event: validation_failed
```

Events are useful for recording something that happened at a particular point in time during the span.

#### Context Propagation

OpenTelemetry uses context propagation to carry tracing context across boundaries.

**why context propagation is Matters**
imagine 

``` markdown
Service A
   ↓
Service B
   ↓
Service C
```

If trace context isn't propagated:
``` markdown
Trace A

Trace B

Trace C
```
here we cannot easily tell that, these operations belongs to the same user request.

with propagation:

``` markdown
Trace ABC
│
├── Service A
│
├── Service B
│
└── Service C 
```

#### Exporters

Now we have generated telemetry. but where does it go?. Thats job of an **Exporter.**

Conceptually:
``` markdown
Application
    ↓
OpenTelemetry SDK
    ↓
Telemetry
    ↓
Exporter
    ↓
Backend
```

For example:

``` markdown
Application
     ↓
OTel
     ↓
Exporter
     ↓
Datadog
```

#### OTLP (Open telemetry protocol.)

It is the protocol used to transmit telemetry between OpenTelemetry components.

Think if it as:
``` markdown
Application
    ↓
Telemetry
    ↓
OTLP
    ↓
Collector / Backend
```

OTLP can carry:
``` markdown
Traces
Metrics
Logs
```


**OTLP vs OpenTelemetry**

OpenTelemetry
The overall observability framework/ecosystem.

OTLP
The protocol used to transmit telemetry.

Simple Analogy:

```markdown
OpenTelemetry = transportation system

OTLP = communication protocol used to transport information
```

#### OpenTelemetry Collector

The OpenTelemetry Collector is a vendor-neutral component that can receive, process, and export telemetry.

Think:

``` markdown
             OTel Collector
                  │
       ┌──────────┼──────────┐
       ↓          ↓          ↓
    Receive     Process    Export
```

* Reciver: Recives the telemetry
* Processor: Modifies or processes telemetry
    processer can be used for things such as:
    batching, filtering, sampling,transforming and enriching telemetry

* Exporter: The collecter's exporter sends telemetry somewhere else.


#### Complete Collector architecture

``` markdown
                    Application
                         │
                         ↓
                    OTLP telemetry
                         │
                         ↓
                ┌──────────────────┐
                │ OTel Collector   │
                │                  │
                │ Receiver         │
                │      ↓           │
                │ Processor        │
                │      ↓           │
                │ Exporter         │
                └────────┬─────────┘
                         │
          ┌──────────────┼──────────────┐
          ↓              ↓              ↓
       Backend A      Backend B      Backend C
```

#### GenAI Semantic Conventions

Semantic conventions are standardized naming/structure conventions for telemetry.

#### Why are GenAI Semantic conventions needed?
LLM Applications have unique operations.
Example: LLM Call

important information could include:

``` markdown
Model
Provider
Input tokens
Output tokens
Operation
Response
Latency
```

A standardized convention helps observability tools understand this information consistently.

##### Example GenAI span

Conceptually:

``` markdown
Span
Name:
chat

Attributes:
model = "..."
provider = "..."
input_tokens = 800
output_tokens = 250
```

The exact semantic convention names evolve, so when implementing production telemetry, you should follow the current OpenTelemetry GenAI semantic-convention specification rather than relying on old tutorials.

#### GenAI Operations

LLM Applications may contain many operations:

for example:
``` markdown
User Request
     ↓
Agent
     ↓
LLM
     ↓
Tool
     ↓
Retriever
     ↓
Database
```

Each can potentially be represented as telemetry.

#### LLM/model operation

example: 

``` markdown
Span:
LLM Generation
```

information:
``` markdown
Model
Provider
Input
Output
Token usage
Latency
Status
```

#### Token usage

One of the most important GenAI metrics.

``` markdown

suppose 

Input tokens = 800
Output tokens = 300
Total:
1100 tokens
```

we can use telemetry to analyze:

``` markdown
Request
 ↓
Input tokens
 ↓
Output tokens
 ↓
Latency
 ↓
Cost
```

This is useful for LLMOps because you want to understand:

* expensive requests
* unusually large prompts
* unusually large outputs
* token growth over time
* model usage


#### Tool Calls

``` markdown
User
 ↓
Agent
 ↓
LLM
 ↓
SQL Tool
 ↓
Database
```

we can create a traces such as:

``` markdown
Trace
│
└── Agent
      │
      ├── LLM
      │
      └── SQL Tool
             │
             └── Database
```

Now with the above flow we can see exactly where time was spent.

#### RAG telemetry

This is especially important for GenAI applications

suppose:

``` markdown
User Question
     ↓
Embedding
     ↓
Vector Database
     ↓
Retrieved Documents
     ↓
LLM
     ↓
Answer
```

Telemetry could represent:

``` markdown
Trace
│
├── Embedding
│
├── Retrieval
│
│   └── Vector DB
│
└── LLM
```

now we can investigate:

``` markdown
Retrieval latency
LLM latency
Number of retrieved documents
Token usage
Errors
```



#### Vendor-Neutral Observability

This is probably the most important architectural idea in this module.

It means your application's telemetry isn't fundamentally tied to one observability vendor.

for example:
``` markdown
                 OpenTelemetry
                       │
        ┌──────────────┼──────────────┐
        ↓              ↓              ↓
    Langfuse        Datadog         MLflow
```

Application produces standardize telemetry. The destination can be changed or expanded.

#### Opentelemetry Mental Model.

``` markdown
                  USER
                    │
                    ↓
             AI APPLICATION
                    │
                    ↓
             INSTRUMENTATION
                    │
                    ↓
            OPEN TELEMETRY SDK
                    │
          ┌─────────┼─────────┐
          ↓         ↓         ↓
       Traces    Metrics     Logs
          │
          ↓
    OTel Collector
          │
    ┌─────┼──────┐
    ↓     ↓      ↓
 Receiver Processor Exporter
                 │
                 ↓
                OTLP
                 │
        ┌────────┼─────────┐
        ↓        ↓         ↓
     Langfuse  Datadog   MLflow
```

and for GenAI Specifically:

``` markdown
GenAI Application
       │
       ↓
OpenTelemetry
       │
       ↓
Trace
       │
       ├── Agent
       │
       ├── LLM
       │    ├── Model
       │    ├── Input tokens
       │    ├── Output tokens
       │    └── Latency
       │
       ├── Retrieval
       │
       ├── Tool
       │
       └── Database
```